### Load data first

In [1]:
import os
import numpy as np
import time
import sys

import pandas as pd
from tqdm import tqdm
from datetime import datetime
from dataclasses import dataclass
from typing import Literal, Annotated


sys.path.append(os.path.abspath(".."))   # Add root path to sys.path
os.chdir("..")  # Change working directory to root path

DATASET_PATH = "data/numpy_grid/"
RESULT_PATH = "data/tracking_output/"

### Setup Parameters

In [4]:
@dataclass
class SensitivityAnalysis:
    max_radius: Annotated[int, "The maximum radius to establish the storm shape vectors"]
    num_radii: Annotated[int, "The number of radii to consider in the storm shape vectors"]
    num_sectors: Annotated[int, "The number of angles (sectors) to divide the storm shape vectors"]
    particle_density: Annotated[float, "The density of particles to sample within the storm areas"]
    max_matching_velocity: Annotated[int, "The maximum velocity (in pixels per hour) to consider for storm movement. Matched pairs exceeding this velocity will be discarded."]
    shape_diff_matching_weight: Annotated[float, "The weight assigned to shape difference in the overall matching score calculation. This value should be between 0 and 1."]

@dataclass
class SensitivityAnalysisResults:
    pod: float
    far: float
    csi: float
    object_consistency: float
    mean_duration: float
    linear_rmse: float
    optimal_tracking: float

def check_override_result(dataset: str, analysis: SensitivityAnalysis) -> pd.DataFrame:
    """
        Implement logic to check if results already exist for the given dataset and analysis parameters
    """
    saved_path = os.path.join(RESULT_PATH, f"{dataset}_sensitivity_analysis_results.csv")
    if os.path.exists(saved_path):
        df = pd.read_csv(saved_path)
    else:
        df = pd.DataFrame(columns=["max_radius", "num_radii", "num_sectors", "particle_density", "max_matching_velocity", "shape_diff_matching_weight", "pod", "far", "csi", 
                                   "object_consistency", "mean_duration", "linear_rmse", "optimal_tracking"])
    
    # Check if the specific analysis parameters already exist in the dataframe
    existing = df[
        (df["max_radius"] == analysis.max_radius) &
        (df["num_radii"] == analysis.num_radii) &
        (df["num_sectors"] == analysis.num_sectors) &
        (df["particle_density"] == analysis.particle_density) &
        (df["max_matching_velocity"] == analysis.max_matching_velocity) &
        (df["shape_diff_matching_weight"] == analysis.shape_diff_matching_weight)
    ]

    return existing


def save_results(dataset: str, analysis: SensitivityAnalysis, results: SensitivityAnalysisResults):
    # Implement logic to save the results to disk
    saved_path = os.path.join(RESULT_PATH, f"{dataset}_sensitivity_analysis_results.csv")
    if os.path.exists(saved_path):
        df = pd.read_csv(saved_path)
    else:
        df = pd.DataFrame(columns=["max_radius", "num_radii", "num_sectors", "particle_density", "max_matching_velocity", "shape_diff_matching_weight", "pod", "far", "csi", 
                                   "object_consistency", "mean_duration", "linear_rmse", "optimal_tracking"])

    new_df = pd.Series({
        "max_radius": analysis.max_radius,
        "num_radii": analysis.num_radii,
        "num_sectors": analysis.num_sectors,
        "particle_density": analysis.particle_density,
        "max_matching_velocity": analysis.max_matching_velocity,
        "shape_diff_matching_weight": analysis.shape_diff_matching_weight,
        "pod": results.pod,
        "far": results.far,
        "csi": results.csi,
        "object_consistency": results.object_consistency,
        "mean_duration": results.mean_duration,
        "linear_rmse": results.linear_rmse,
        "optimal_tracking": results.optimal_tracking
    })

    # Override the existing rows if they exist
    existing = check_override_result(dataset, analysis)
    if existing is not None and not existing.empty:
        df = df[~df.index.isin(existing.index)]

    df.loc[len(df)] = new_df
    df.to_csv(saved_path, index=False)

In [5]:
def run_sensitivity_analysis(dataset: str, analysis: SensitivityAnalysis) -> SensitivityAnalysisResults:
    """
        Main entries to run sensitivity analysis. Phases include:
            1. Load dataset and configure model with given parameters
            2. Identify storms in each frame
            3. Perform and evaluate nowcasting across time
            4. Evaluation of tracking performance
    """
    # Phase 1: Load dataset and configure model
    from src.preprocessing import read_numpy_grid, nexrad_numpy_preprocessing_pipeline

    source_path = os.path.join(DATASET_PATH, dataset)
    img_paths = [os.path.join(source_path, img_name) for img_name in sorted(os.listdir(source_path)) if img_name.endswith('.npy')]
    dbz_maps: list[tuple[np.ndarray, datetime]] = []

    for path in tqdm(img_paths, desc="Processing images and detecting storms"):
        file_name = path.split("/")[-1].split(".")[0]

        time_frame = datetime.strptime(file_name[4:19], "%Y%m%d_%H%M%S")
        img = read_numpy_grid(path)
        dbz_maps.append((img, time_frame))

    from src.models import OursPrecipitationModel
    from src.identification import MorphContourIdentifier

    model = OursPrecipitationModel(identifier=MorphContourIdentifier(), max_velocity=analysis.max_matching_velocity,
                                   weights=(1 - analysis.shape_diff_matching_weight, analysis.shape_diff_matching_weight),
                                   radii=np.linspace(start=analysis.max_radius/analysis.num_radii, stop=analysis.max_radius, num=analysis.num_radii, dtype=int).tolist(),
                                   num_sectors=analysis.num_sectors, density=analysis.particle_density)
    
    # Phase 2: Identify storms
    storms_maps = []
    for idx, (dbz_map, time_frame) in tqdm(list(enumerate(dbz_maps)), desc="Identifying storms in each frame"):
        storms_map = model.identify_storms(dbz_map, time_frame, map_id=f"time_{idx}", threshold=35, filter_area=50)
        storms_maps.append(storms_map)

    # Phase 3: Evaluate Nowcasting across time
    from src.cores.metrics import PredictionBenchmarkModel

    ours_model_evaluation = PredictionBenchmarkModel()
    temp_storm_map = storms_maps
    SLOW_START_STEPS = 3
    PREDICT_FORWARD_STEPS = 3

    for i in range(SLOW_START_STEPS):
        model.processing_map(temp_storm_map[i])  # Warm-up phase

    for curr_map, future_map in tqdm(list(zip(temp_storm_map[SLOW_START_STEPS:], temp_storm_map[PREDICT_FORWARD_STEPS + SLOW_START_STEPS:])), desc="Predicting precipitation maps"):
        # Predict map using current data
        dt_seconds = (future_map.time_frame - model.storms_maps[-1].time_frame).total_seconds()
        predicted_map = model.forecast(dt_seconds)
        ours_model_evaluation.evaluate_predict(future_map, predicted_map)
        model.processing_map(curr_map)  # Update model with the current map

    # Phase 4: Evaluate tracking performance
    from src.cores.metrics import PostEventClustering, linear_tracking_error

    # 4.1 Object consistency first
    object_consistency_scores = []
    for track in model.tracker.tracks:
        areas = [storm.contour.area for storm in track.storms.values()]
        area_changes = [abs(areas[i] - areas[i - 1]) / areas[i - 1] for i in range(1, len(areas)) if areas[i - 1] != 0]
        object_consistency_scores.append(np.mean(area_changes) if area_changes else 0)

    object_consistency_score = np.mean(object_consistency_scores)

    # 4.2 Mean duration of tracked objects
    mean_duration = np.mean([len(track.storms) for track in model.tracker.tracks])

    # 4.3 Linear RMSE
    linear_error_lsts = [linear_tracking_error(storm.history_movements[:-1]) ** 2 for storms_map in storms_maps 
                                                                    for storm in storms_map.storms if len(storm.history_movements) >= mean_duration]
    linear_rmse = np.sqrt(np.mean(linear_error_lsts))

    # 4.4 Optimal tracking evaluation
    centroids = []
    clusters_assigned = []

    FIRST_TIME_FRAME = dbz_maps[0][1]

    def _convert_time_frame_to_seconds(time_frame: datetime) -> float:
        return time_frame.timestamp() - FIRST_TIME_FRAME.timestamp()

    for track_history in model.tracker.tracks:
        track_centroids = [(storm.centroid[0], storm.centroid[1], _convert_time_frame_to_seconds(time_frame)) for time_frame, storm in track_history.storms.items()]
        centroids.extend(track_centroids)
        clusters_assigned.extend([track_history.id] * len(track_centroids))
    
    centroids = np.array(centroids)

    postevent_analysis = PostEventClustering(centroids, max_window_time=600, spatial_distance_threshold=50)
    reassigned_clusters_centers = postevent_analysis.fit_transform(num_clusters=len(model.tracker.tracks), clusters_assigned=clusters_assigned, max_epochs=50)

    merged_clusters_assigned = [postevent_analysis.clusters_merged_dict.get(i, i) for i in clusters_assigned]
    score_lst = [1 if merged_clusters_assigned[i] == reassigned_clusters_centers[i] else 0.5 if reassigned_clusters_centers[i] != -1 else 0 for i in range(len(clusters_assigned))]

    optimal_tracking_score = np.mean(score_lst)

    return SensitivityAnalysisResults(
        pod=np.mean(ours_model_evaluation.pods),
        far=np.mean(ours_model_evaluation.fars),
        csi=np.mean(ours_model_evaluation.csis),
        object_consistency=object_consistency_score,
        mean_duration=mean_duration,
        linear_rmse=linear_rmse,
        optimal_tracking=optimal_tracking_score
    )

## Running the test

In [6]:
import itertools

param_grid = {
    "dataset": ["KARX"], 
    "max_radius": [20, 40, 60, 80, 100, 120],
    "num_radii": [4],
    "num_sectors": [6],
    "particle_density": [0.05],
    "max_matching_velocity": [200],
    "shape_diff_matching_weight": [0.5]
}

keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

In [7]:
results = []
for idx, combo in enumerate(combinations):
    print("-" * 50)
    print(f"Running combination {idx + 1}/{len(combinations)}: {combo}")
    analysis_params = SensitivityAnalysis(**{k: v for k, v in combo.items() if k in SensitivityAnalysis.__dataclass_fields__})

    if not check_override_result(combo["dataset"], analysis_params).empty:
        print("Skipping existing result.")
        continue

    results.append(run_sensitivity_analysis(combo["dataset"], analysis_params))
    save_results(combo["dataset"], analysis_params, results[-1])

--------------------------------------------------
Running combination 1/6: {'dataset': 'KARX', 'max_radius': 20, 'num_radii': 4, 'num_sectors': 6, 'particle_density': 0.05, 'max_matching_velocity': 200, 'shape_diff_matching_weight': 0.5}

## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



Predicting precipitation maps: 100%|██████████| 87/87 [00:01<00:00, 48.03it/s] 


Post-event clustering:   0%|          | 0/50 [00:00<?, ?it/s]

--------------------------------------------------
Running combination 2/6: {'dataset': 'KARX', 'max_radius': 40, 'num_radii': 4, 'num_sectors': 6, 'particle_density': 0.05, 'max_matching_velocity': 200, 'shape_diff_matching_weight': 0.5}


Predicting precipitation maps: 100%|██████████| 87/87 [00:02<00:00, 36.85it/s] 


...

Post-event clustering:   0%|          | 0/50 [00:00<?, ?it/s]

--------------------------------------------------
Running combination 5/6: {'dataset': 'KARX', 'max_radius': 100, 'num_radii': 4, 'num_sectors': 6, 'particle_density': 0.05, 'max_matching_velocity': 200, 'shape_diff_matching_weight': 0.5}


Predicting precipitation maps: 100%|██████████| 87/87 [00:01<00:00, 48.58it/s] 


Post-event clustering:   0%|          | 0/50 [00:00<?, ?it/s]

--------------------------------------------------
Running combination 6/6: {'dataset': 'KARX', 'max_radius': 120, 'num_radii': 4, 'num_sectors': 6, 'particle_density': 0.05, 'max_matching_velocity': 200, 'shape_diff_matching_weight': 0.5}
Skipping existing result.


## Load model